[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-bigmartsales.ipynb)

# Full Project: Sales Prediction of a Retailer

*AIBits Academy · Machine Learning End To End · Full Project*

8,523 products across 10 outlets — messy real-world missing data, and a Random Forest that beats linear regression by explaining 10 more percentage points of variance.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

def fetch(url, target, member=None):   # public source; a zip member is extracted and renamed to `target`
    if os.path.exists(target):
        return
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    blob = urllib.request.urlopen(req, timeout=120).read()
    if member:
        blob = zipfile.ZipFile(io.BytesIO(blob)).read(member)
    open(target, 'wb').write(blob)
    print('downloaded', target)

fetch('https://raw.githubusercontent.com/akki8087/Big-Mart-Sales/master/Train.csv', 'Train.csv')

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A retail chain — the same problem faced by any multi-outlet Indian retailer like a regional supermarket chain in Gujarat — has collected historical sales for thousands of products across its outlets. It wants to predict sales for each product at each outlet, both to plan inventory and to understand which product and outlet attributes actually drive sales.

> **Dataset**
>
> **8,523 training rows across 1,559 products and 10 outlets**, with item attributes (weight, fat content, visibility, type, MRP) and outlet attributes (establishment year, size, location tier, type). Target: `Item_Outlet_Sales`. [Dataset source →](https://raw.githubusercontent.com/akki8087/Big-Mart-Sales/master/Train.csv)

## Step 1 — The Missing-Data Problem

In [ ]:
import pandas as pd
train = pd.read_csv('Train.csv')
print(train.isnull().sum()[train.isnull().sum()>0])

Two very different missing-data situations. `Item_Weight` is missing seemingly at random, but the same product (`Item_Identifier`) always has the same weight across every outlet it's sold in — so a per-item average, not a global average, is the right imputation. `Outlet_Size` is missing for entire outlets, so it's imputed from the most common size within the same `Outlet_Type` instead.

In [ ]:
# Item_Weight: impute from the same item's weight elsewhere
item_avg_weight = train.groupby('Item_Identifier')['Item_Weight'].mean()
mask = train['Item_Weight'].isnull()
train.loc[mask,'Item_Weight'] = train.loc[mask,'Item_Identifier'].map(item_avg_weight)

# Outlet_Size: impute from the mode within the same Outlet_Type
mode_by_type = train.groupby('Outlet_Type')['Outlet_Size'].agg(lambda x: x.mode()[0])
mask = train['Outlet_Size'].isnull()
train.loc[mask,'Outlet_Size'] = train.loc[mask,'Outlet_Type'].map(mode_by_type)

## Step 2 — Cleaning Inconsistent Categories

A classic data-quality issue hiding in plain sight:

In [ ]:
print(train['Item_Fat_Content'].value_counts())

What looks like 5 categories is really 2, with inconsistent labelling from data entry (`LF`, `low fat`, and `Low Fat` are the same thing). Collapsing these before modelling matters — leaving them separate would force the model to learn the same relationship five times over with less data each time.

## Step 3 — Feature Engineering & Model Comparison

Beyond cleanup, one engineered feature stands out: `Outlet_Age` (2013 − establishment year) is far more directly interpretable to a model than a raw calendar year. Categorical columns are label-encoded, and three regression approaches are compared:

The lesson's next step (elided in its snippet): collapse the fat-content labels, engineer `Outlet_Age`, label-encode the categoricals and split 80/20.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

train['Item_Fat_Content'] = train['Item_Fat_Content'].replace({'LF': 'Low Fat', 'low fat': 'Low Fat', 'reg': 'Regular'})
train['Item_Weight'] = train['Item_Weight'].fillna(train['Item_Weight'].mean())   # items with no weight anywhere: global mean
train['Outlet_Age'] = 2013 - train['Outlet_Establishment_Year']
features = [c for c in train.columns if c not in ('Item_Outlet_Sales', 'Item_Identifier', 'Outlet_Identifier', 'Outlet_Establishment_Year')]
X = train[features].copy()
for col in X.select_dtypes(exclude='number').columns:
    X[col] = LabelEncoder().fit_transform(X[col])
y = train['Item_Outlet_Sales']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "train /", len(X_test), "test;", len(features), "features")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# ... encode categoricals, engineer Outlet_Age, split 80/20 ...
for name, model in [('LinearRegression', LinearRegression()),
                    ('Ridge', Ridge(alpha=1.0)),
                    ('RandomForest', RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=50))]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print(f"{name:16s} RMSE={np.sqrt(mean_squared_error(y_test,pred)):.2f}  R²={r2_score(y_test,pred):.4f}")

Random Forest wins clearly — explaining **62% of sales variance** versus 52% for the linear approaches, an RMSE improvement of roughly ₹124 per prediction. Ridge's near-identical performance to plain Linear Regression (0.5208 vs 0.5207 R²) tells you multicollinearity isn't the bottleneck here — the real limitation of the linear models is their inability to capture non-linear and interaction effects that Random Forest picks up naturally.

## Step 4 — What Actually Drives Sales?

The Random Forest is the last model fitted in the loop above.

In [ ]:
rf = model

In [ ]:
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
print(importances.head())

Two features — **Item_MRP** (list price) and **Outlet_Type** — together account for over 93% of the Random Forest's predictive power. Everything else the dataset provides (item weight, fat content, visibility, product category, outlet location tier) contributes almost nothing incrementally. This is a genuinely useful, if slightly deflating, business finding: most of the carefully-collected product-level attributes barely move the needle once price and outlet format are known.

## Visualizing the 93% Concentration

The bottom three features barely register next to the top two.

> **💡 A finding that changes what data collection to prioritize next**
>
> When two features dominate this completely, it's worth asking whether the *next* data-collection effort should focus on getting more outlet-format and pricing granularity right, rather than collecting yet more product-level attributes that this analysis suggests barely matter. This is exactly the kind of feature-importance-driven prioritization decision covered on the Model Interpretability & Explainability page.

## Key Business Takeaways

- Random Forest (R²=0.6192) outperforms both Linear Regression and Ridge (R²≈0.521) by a wide margin — the relationship between these features and sales is meaningfully non-linear.
- Item_MRP and Outlet_Type together explain over 93% of the model's predictive power — most other collected attributes (weight, fat content, location tier) add almost nothing once these two are known.
- Two different missing-data mechanisms (Item_Weight missing at the row level, Outlet_Size missing at the outlet level) required two different imputation strategies — a single blanket approach would have been wrong for at least one of them.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · How much is missing?

Re-read the raw file into `raw` and store the percentage of missing `Item_Weight` values in `weight_missing_pct` (about 17.2%).

In [ ]:
import pandas as pd
raw = pd.read_csv("Train.csv")
weight_missing_pct = None   # TODO


In [ ]:
try:
    check("about 17.2%", abs(weight_missing_pct - 17.17) < 0.05)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
raw = pd.read_csv("Train.csv")
weight_missing_pct = float(raw["Item_Weight"].isna().mean() * 100)

```

</details>

### Exercise 2 · Medium · Which outlet type sells most?

Store the mean `Item_Outlet_Sales` per `Outlet_Type` in `sales_by_type` (a Series) and the best type's name in `best_type`.

In [ ]:
sales_by_type = best_type = None   # TODO (use the cleaned `train`)


In [ ]:
try:
    ref = train.groupby("Outlet_Type")["Item_Outlet_Sales"].mean()
    check("means", abs(sales_by_type - ref).max() < 1e-9)
    check("best type", best_type == ref.idxmax())
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
sales_by_type = train.groupby("Outlet_Type")["Item_Outlet_Sales"].mean()
best_type = sales_by_type.idxmax()

```

</details>

### Exercise 3 · Stretch · Two features do the work

Using the lesson's fitted random forest `rf`, store the names of the two most important features (as a set) in `top2`, and their combined importance share in `top2_share`.

In [ ]:
top2 = top2_share = None   # TODO


In [ ]:
try:
    check("Item_MRP and Outlet_Type", top2 == {"Item_MRP", "Outlet_Type"})
    check("over 85% together", top2_share > 0.85)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
imp = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=False)
top2 = set(imp.index[:2])
top2_share = float(imp.iloc[:2].sum())

```

When two features carry >90% of the signal, most of the dataset's other columns are nearly irrelevant to a price/outlet model.

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Sales Prediction of a Retailer**.*